In [ ]:
import math
import random

from hmmlearn import hmm
import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
from IPython.display import Audio

pachabel_melody = [
    [50, 1],
    [45, 1],
    [47, 1],
    [42, 1],
    [43, 1],
    [38, 1],
    [43, 1],
    [45, 1],
    # 1st violin start
    [78, 1],
    [76, 1],
    [74, 1],
    [73, 1],
    [71, 1],
    [69, 1],
    [71, 1],
    [73, 1],
    [74, 1],
    [73, 1],
    [71, 1],
    [69, 1],
    [67, 1],
    [66, 1],
    [67, 1],
    [64, 1],
    [62, 0.5],
    [66, 0.5],
    [69, 0.5],
    [67, 0.5],
    [66, 0.5],
    [62, 0.5],
    [66, 0.5],
    [64, 0.5],
    [62, 0.5],
    [59, 0.5],
    [62, 0.5],
    [69, 0.5],
    [67, 0.5],
    [71, 0.5],
    [69, 0.5],
    [67, 0.5],
    [66, 0.5],
    [62, 0.5],
    [64, 0.5],
    [73, 0.5],
    [74, 0.5],
    [78, 0.5],
    [81, 0.5],
    [69, 0.5],
    [71, 0.5],
    [67, 0.5],
    [69, 0.5],
    [66, 0.5],
    [62, 0.5],
    [74, 0.5],
    [74, 0.75],
    [73, 0.25],
]

fur_elise_melody = [
    # Measure 1 (Pickup)
    [76, 0.5], # E5
    [75, 0.5], # D#5

    # Measure 2
    [76, 0.5], # E5
    [75, 0.5], # D#5
    [76, 0.5], # E5
    [71, 0.5], # B4
    [74, 0.5], # D5
    [72, 0.5], # C5

    # Measure 3
    [69, 1.0], # A4
    [60, 0.5], # C4
    [64, 0.5], # E4
    [69, 0.5], # A4

    # Measure 4
    [71, 1.0], # B4
    [64, 0.5], # E4
    [68, 0.5], # G#4
    [71, 0.5], # B4

    # Measure 5
    [72, 1.0], # C5
    [64, 0.5], # E4
    [76, 0.5], # E5
    [75, 0.5], # D#5

    # Measure 6
    [76, 0.5], # E5
    [75, 0.5], # D#5
    [76, 0.5], # E5
    [71, 0.5], # B4
    [74, 0.5], # D5
    [72, 0.5], # C5

    # Measure 7
    [69, 1.0], # A4
    [60, 0.5], # C4
    [64, 0.5], # E4
    [69, 0.5], # A4

    # Measure 8
    [71, 1.0], # B4
    [64, 0.5], # E4
    [72, 0.5], # C5
    [71, 0.5], # B4

    # Measure 9
    [69, 1.5], # A4
    # [76, 0.5], # E5
    # [75, 0.5], # D#5

    # Measure 10
    # [76, 0.5], # E5
    # [75, 0.5], # D#5
    # [76, 0.5], # E5
    # [71, 0.5], # B4
    # [74, 0.5], # D5
    # [72, 0.5]  # C5
]



def melody_to_audio(melody):
    midi = pretty_midi.PrettyMIDI()
    # 0=piano, 40=violin
    instrument = pretty_midi.Instrument(program=40)

    t = 0
    for pitch, duration in melody:
        instrument.notes.append(
            pretty_midi.Note(
                velocity=100,
                pitch=pitch,
                start=t,
                end=t + duration
            )
        )
        t += duration

    midi.instruments.append(instrument)

    audio = midi.synthesize(fs=44100)
    return Audio(audio, rate=44100)

def melody_to_samples(melody, samples_per_step, noise_scale=1):
    output = []
    for pitch, duration in melody:
        samples = duration * samples_per_step
        samples = int(samples) + int(random.random() < (samples - int(samples)))
        output.extend([pitch] * samples)

    output = np.array(output) + np.random.normal(scale=noise_scale, size=len(output))
    return output

melodies = {1 : pachabel_melody, 2 : fur_elise_melody}

def encode_melody(melody):
    notes = np.asarray(melody, dtype=float)

    pitches, durations = notes.T
    n_notes = len(notes)
    model = hmm.GaussianHMM(
        n_components=n_notes, covariance_type="diag",
        init_params="", params="", algorithm="viterbi"
    )

    model.startprob_ = durations / durations.sum()
    model.means_ = pitches.reshape(-1, 1).copy()

    model.covars_ = np.ones((n_notes, 1))

    advance = 1.0 / np.maximum(10.0 * durations, 1.0)
    model.transmat_ = np.zeros((n_notes, n_notes))
    indices = np.arange(n_notes)
    model.transmat_[indices, indices] = 1.0 - advance
    
    model.transmat_[indices, (indices + 1) % n_notes] += advance
    model.state_offsets_ = np.concatenate(([0.0], np.cumsum(durations[:-1])))
    return model


test_samples = {
  "1": [
    43.02369283911827,
    42.18755886327329,
    42.50574444173104,
    44.79627064244632,
    43.174795951946024,
    43.19680797342837,
    44.29290478318237,
    42.53229125499459,
    42.86822113927452,
    43.84780432675458,
    38.920587193036916,
    37.4466381583145,
    37.192975360981876,
    36.54666440320561,
    38.55730835737986,
    37.51861477117201,
    36.91870522652686,
    37.719891871161884,
    38.15951042169094,
    37.101168223780554,
    42.27052169607325,
    44.65791033837104,
    44.06942695506933,
    43.51317105581712,
    42.19761100982326,
    42.85927664240546,
    41.6481391558641,
    42.29991601958541,
    43.44757754565237,
    44.38601470536294,
    43.86080982017466,
    45.032604144765905,
    43.43700147866673,
    46.63676031945879,
    44.99495058768235,
    43.61691690278979,
    44.69083610589367,
    46.13710894915338,
    44.89106821259454,
    45.33769066912908,
    78.19290946859459,
    76.97965316152926,
    78.66007160299448,
    78.53903752901039,
    77.88934048009982,
    74.48487929228085,
    78.10922689820781,
    78.33616859288517,
    79.45603295661465,
    80.44467570166172,
    76.13008199650238,
    75.55948797024777,
    76.17345026982983,
    76.69923953106725,
    75.26142954394237,
    77.61848347427276,
    75.70969238000986,
    77.42089869899625,
    76.7625837178756,
    76.25657759081393,
    72.50091709790152,
    75.06643149492402,
    75.65769331603306,
    74.62094654247858,
    71.94054985395442,
    74.0726415225135,
    73.75192508215922,
    73.109199275876,
    73.50190928815842,
    74.23165512852124,
    73.00462482348729,
    73.7355221867947,
    73.92205880795103,
    72.30360467950402,
    72.19484544075866,
    72.97793162742651,
    72.34063610456109,
    73.21373775073461,
    72.70925331562326,
    74.45095227975561,
    70.77614580277574,
    71.4806433083937,
    72.01754408372628,
    70.43813488303661,
    70.4528582691922,
    71.87771947194729,
    71.88790560069303,
    71.97890648238325,
    70.40011448413946,
    72.33864804120758,
    68.04723416095163,
    70.87180819248033,
    69.0844194889221,
    69.45690289529202,
    69.20423906014088,
    69.47798772994388,
    68.23978290497078,
    70.53595785821689,
    67.45114607890127,
    68.91102534201536
  ],
  "2": [
    77.23453902774202,
    75.3789403535647,
    74.95254896378228,
    74.47544642198696,
    75.69113317345652,
    70.61402191639095,
    69.9162093861773,
    70.3798753567574,
    70.10682343625253,
    70.78853767473056,
    74.40843185559613,
    73.84758750486692,
    74.55784383612718,
    74.21856567390857,
    75.098411251918,
    71.22747542057142,
    72.76447846814247,
    74.11906976946575,
    72.94678754316499,
    71.00222721822942,
    69.93524553097747,
    67.49112490168588,
    70.43362986263791,
    69.89213590639754,
    70.70014677500622,
    68.83169565425929,
    70.03117605985635,
    69.16814901827779,
    69.46207893588,
    67.42066005155097,
    61.080159980788764,
    58.984731011258226,
    58.61879694851043,
    58.38081419786949,
    60.385553657928945,
    64.63867645386092,
    64.1158177314781,
    66.02388757765762,
    63.76960218713839,
    64.11454657648278,
    66.91214588515167,
    68.17510103569165,
    67.40002722572866,
    68.08530252426641,
    69.03680715936113,
    71.42267473273712,
    71.83860629525935,
    71.75021071663406,
    69.4726320877536,
    69.96247459962862,
    72.38529809279642,
    72.97743905211908,
    70.94943102189393,
    71.29957892731646,
    68.95250200173022,
    63.7002951090539,
    64.91272023064029,
    63.07065422257291,
    64.06731250655741,
    64.74695056198833,
    67.77694898366511,
    68.75655462642308,
    68.17493509767503,
    68.62363910441304,
    68.20223386373556,
    72.39590928030906,
    69.24253130169949,
    71.50116260905047,
    71.10898661117479,
    72.61257422167411,
    72.19414725349286,
    73.0360539342957,
    70.94331775694872,
    71.35916555241447,
    71.00317265887763,
    72.87727156908862,
    74.17712160004092,
    70.88845574488828,
    71.09577588530557,
    72.1282251541821,
    64.99526957610294,
    64.77337706182855,
    64.40826887017238,
    62.94910012282522,
    64.0812337775115,
    77.45754036658487,
    75.49746083821348,
    76.87290720097762,
    77.10131404735638,
    76.08769614356703,
    73.93597059526118,
    75.18994163201862,
    73.53134386961844,
    75.09799662016088,
    74.50494875931541,
    75.03820505979715,
    76.67623302556908,
    77.45487800634388,
    76.33165046044148,
    77.34227575803348
  ],
  "3": [
    68.86248352391377,
    70.7082636024802,
    68.85419133872287,
    69.47953771861707,
    68.78729673168088,
    71.38758252904015,
    70.65199782456236,
    71.06921068044757,
    72.41947291103816,
    70.06300134166146,
    69.3138905320603,
    70.10074973268138,
    70.67432816597005,
    72.208752443362,
    71.69382655791338,
    64.77062167093354,
    65.51806038037581,
    64.24474814700311,
    63.63871515781874,
    64.82260665351261,
    69.76697709180931,
    67.09484695668513,
    68.57117644422434,
    67.53350978413006,
    68.62861147627102,
    71.07498893597992,
    71.200459176022,
    70.47954391435181,
    71.53813524960232,
    69.39337657056477,
    72.25610499914085,
    71.09735239296793,
    71.57116422371757,
    71.83850336453591,
    72.9527141311925,
    71.75820112469194,
    72.81547221881361,
    71.60546219008585,
    73.25149464170168,
    72.84658200945827,
    64.22731352750702,
    63.13201554990621,
    63.66957030129812,
    64.17638075581476,
    64.26803668751579,
    74.81615824988033,
    76.49640640680012,
    77.75905183253639,
    75.17015410130979,
    76.42390806522312,
    74.99346735116107,
    75.6073226675666,
    76.80524369696386,
    73.81642538929042,
    73.77614501575395,
    74.16348485758648,
    76.42568675891341,
    75.48400390687503,
    76.21929787339383,
    76.61619692072684,
    75.66738897881189,
    75.79993461055763,
    75.14108748999608,
    75.42652180204117,
    74.39176518985535,
    74.6297320937725,
    75.51073638214275,
    75.80895282229457,
    77.34836271352076,
    77.44663489846083,
    74.0869055878442,
    70.83743153688269,
    71.02056412962797,
    71.25761025817326,
    70.24972220440588,
    72.92264606061568,
    72.9551674676758,
    75.30543850342792,
    76.17888129158226,
    74.6000865392378,
    73.10286693009188,
    71.917069576122,
    73.04314433487168,
    71.94658107103406,
    74.29012903730052,
    69.34819877426793,
    68.58163140998133,
    70.64696687332192,
    68.82119778757234,
    68.44618560353918,
    69.08985029046335,
    69.82202334225036,
    69.0544077793157,
    68.47689009940154,
    69.39938315351398,
    59.733167842280395,
    59.47208573268553,
    57.27493690101673,
    59.05159565902495,
    60.1587828671593
  ],
  "4": [
    68.39059951310672,
    68.22071055667654,
    68.01325968537729,
    68.24850863957334,
    67.1805365997038,
    69.28633649398748,
    69.42068256107994,
    67.38840811631053,
    67.65514708766298,
    67.59287163781416,
    71.80099557993888,
    71.3386805699115,
    71.52990075393019,
    69.10681565294331,
    69.38781166311104,
    70.5355533907225,
    72.40921408382258,
    71.94005829081333,
    71.34850508831845,
    71.72884885636945,
    72.6812552000297,
    73.9541094907253,
    72.8405628340224,
    72.50986740291708,
    73.4711742598444,
    73.02897075490613,
    74.51372658759641,
    69.90404923869932,
    71.3945853797107,
    72.12401989991967,
    74.79458605806163,
    74.48071768432321,
    75.42751889781447,
    74.21862661516946,
    72.9133688879404,
    74.84361060839723,
    73.51247897621221,
    72.95342184677484,
    74.75182105180602,
    74.52800235894631,
    72.91490749753298,
    71.73937834245405,
    72.67324586876255,
    72.80330260113725,
    71.9603587250106,
    71.84704274948128,
    73.00231398668664,
    73.82385961822044,
    72.98441076598458,
    71.80778196297635,
    70.04935116549869,
    72.77821908831896,
    70.86119926363385,
    70.90662916617974,
    71.82742213482835,
    71.95478755750196,
    70.55624890076578,
    72.42090155722414,
    71.71186348267037,
    69.48214257279628,
    68.57157942486975,
    69.01953082262,
    68.90627350096582,
    69.93435809364149,
    70.43760702513147,
    69.03098182140295,
    67.7184692868973,
    66.61149633530204,
    68.9315812173822,
    68.66594858196945,
    64.91102179944123,
    66.80486546407869,
    68.00333668499331,
    66.665104482564,
    66.47316529274629,
    67.33373675138522,
    66.5225201881511,
    66.95279709936348,
    68.01467103278918,
    66.43525185061597,
    66.90061806361366,
    65.58270354652888,
    67.10624308366725,
    67.69187363629959,
    65.63789216875293,
    65.77663655087254,
    66.03210787500524,
    64.86510581530726,
    65.10762083946851,
    65.02877091270157,
    67.72800934374278,
    65.59211572280192,
    65.66483293863419,
    65.05573879489262,
    67.6348148017737,
    66.13448879711618,
    66.91520670167336,
    67.41144208354004,
    66.5883131453802,
    66.17834901467064
  ],
  "5": [
    37.0375174436813,
    38.107121254131854,
    36.96280787758621,
    37.62763661287787,
    37.77955592141688,
    40.16191120685965,
    38.17064898125487,
    40.00411932458192,
    38.002345414857125,
    38.27809256812273,
    42.90338648560116,
    43.08094290581461,
    44.03274825253737,
    42.78781472683989,
    40.728103134058,
    43.679474387237356,
    45.36428748336507,
    43.75466707349297,
    43.41157394227269,
    42.83620258226355,
    45.19639690178533,
    44.12766642929502,
    44.711968003835636,
    44.78964269564844,
    45.72960234323611,
    44.87528152669739,
    44.68558978181466,
    44.388695205613054,
    44.656994330401716,
    44.65269182353999,
    76.57216327073999,
    80.0181095958332,
    76.97990628721617,
    75.84707610205353,
    78.04970871806866,
    79.54125393023254,
    77.02771402119288,
    76.91999179914541,
    79.17163581921082,
    77.01239847629323,
    77.06629138429487,
    77.4431959179651,
    74.95994502028641,
    77.99246483972306,
    75.79709173439495,
    75.5625311690347,
    75.70714320419852,
    74.35783883487225,
    74.78154850989822,
    74.88662729716692,
    75.04996314541513,
    73.10448957718408,
    73.0799767992909,
    74.3483559854227,
    72.8463358644029,
    73.14980672444648,
    74.62650369065172,
    74.38716954701349,
    74.46131809795766,
    74.16261025891862,
    72.70301679157907,
    72.20723229806693,
    71.92030955033145,
    71.59182873247168,
    71.62435056317356,
    71.97304990534249,
    72.86810310725686,
    72.77922148945308,
    72.48642993613868,
    73.07958343012874,
    70.75405405621821,
    72.06946005439467,
    71.82894025297773,
    71.59643869169099,
    69.69539426138002,
    70.43321750659096,
    72.95592944149726,
    71.52275711156837,
    70.97759983073455,
    71.53052893619441,
    68.92679817765735,
    67.83257657858456,
    67.93786378316334,
    67.66958070787557,
    68.29087378079211,
    70.29487715181169,
    66.45059798802586,
    67.94316246408557,
    69.94865502690355,
    69.16347667939823,
    70.04796597077163,
    68.64292140007139,
    71.85032658078399,
    69.0132629572033,
    70.394281451939,
    70.74079290741406,
    71.97560459479446,
    70.12392838448997,
    70.89436233123409,
    71.96497649371562
  ],
  "6": [
    75.35450423033164,
    75.82620413030598,
    77.07917249499405,
    77.2771029283192,
    75.22793178570026,
    76.92351989329521,
    76.53618485241573,
    74.34418893821436,
    74.59051587518758,
    74.86605891349716,
    75.65230349109582,
    75.97981152908402,
    75.79974490759352,
    73.98160401102413,
    73.1187073601806,
    73.10222662462523,
    72.62807841087378,
    73.67271082742205,
    74.98101424504604,
    73.5969257058768,
    75.94352713863807,
    74.00548008762199,
    74.68745270558586,
    73.04740657556621,
    73.37907913795482,
    72.42535720454408,
    70.94926520969548,
    77.03305017554781,
    71.34353662688449,
    71.42301780140713,
    73.27249608973202,
    72.35437875362072,
    71.77202049526241,
    73.28358233412335,
    72.2331770582369,
    74.47664725990886,
    69.72155417873597,
    70.67354583130935,
    71.52301384279998,
    72.38520878971669,
    71.32048653411562,
    72.21829796752043,
    69.63321417511602,
    71.18442594470729,
    70.63480490099398,
    70.37126710352678,
    70.96823060539386,
    70.51452191601312,
    69.04071938365948,
    69.59718629606003,
    68.94999680300906,
    67.25726313196576,
    69.04983780895293,
    68.90049017797718,
    70.37273222188645,
    67.32981892827185,
    69.47389976813155,
    69.40594302344546,
    66.69457392090148,
    67.37482849292516,
    71.28983715789266,
    70.36254768459482,
    72.67785030252696,
    72.06529125799506,
    70.12078037828678,
    71.50191805235418,
    72.29202755824682,
    70.4553233950516,
    70.39050600565014,
    70.33676745031659,
    70.32950626057486,
    70.100121567196,
    72.64534012348787,
    73.61486674117711,
    72.37353602155437,
    74.44991358516478,
    73.46540649914866,
    74.43152706290194,
    73.79687122013941,
    74.37569490352708,
    74.1213552379442,
    73.80167399537763,
    72.17100299339933,
    72.74812819841732,
    72.8831988077319,
    75.18692434509506,
    75.52737471393382,
    73.12486188679888,
    75.34836111140163,
    74.42677740767488,
    73.27790333202712,
    73.25492515259799,
    74.38992645163594,
    73.15689207575768,
    73.96210329402524,
    74.12807738240797,
    73.84103176736802,
    72.0385371488845,
    73.27931545022628,
    74.11397605796908,
    73.00107167357949,
    71.52889191565208,
    71.51847799519608,
    73.18045205939949,
    73.11327692315128,
    72.9729660291386,
    73.34530609068736,
    72.00395501798678,
    71.01146018058924,
    70.91522357846117,
    70.30525060721331,
    69.9303695635021,
    71.7614540402256,
    70.35553419556568,
    71.2478674391066,
    70.76877145291343,
    71.15523278830236,
    71.01064441526127,
    72.38085667874928,
    71.4470881000349
  ],
  "7": [
    70.14053497400714,
    72.1498126583953,
    71.71449027564773,
    69.83339899156448,
    64.3329864669989,
    63.413811853202496,
    62.79823921181302,
    63.262471584923254,
    67.58349429257622,
    69.3088983543327,
    68.3056280715104,
    67.1685229265797,
    70.80285095385351,
    71.21111565127588,
    71.56814365356087,
    70.30504828153795,
    71.56537901994515,
    72.691533681298,
    71.30266701279625,
    72.35563440347268,
    71.22855208779046,
    73.15114064100707,
    70.17995188825105,
    73.10264965057895,
    63.27089663287603,
    65.14375448554789,
    64.09982517015506,
    64.17124735957809,
    75.67211702325922,
    75.32656982595563,
    75.63247259817636,
    76.93310282111615,
    74.87924832586067,
    75.5321053685425,
    74.36689830652836,
    75.95678330068402,
    77.22044236865091,
    74.34557319999135,
    76.0291938240831,
    77.13933144655417,
    74.7670833110275,
    74.62976985535833,
    74.39118540266162,
    75.97391439549216,
    75.69279626700433,
    75.59567266572893,
    76.04664814173515,
    77.79883728116813,
    73.05216877391999,
    71.42595369624388,
    70.18363294371953,
    70.0492275367283,
    73.61419604207326,
    74.12353189732612,
    73.3461535806195,
    73.50854767079643,
    71.21073852821806,
    72.27816959540255,
    70.7068430977366,
    71.7580103746681,
    68.29264016182361,
    69.32061745363445,
    68.99821636900423,
    68.5250167920876,
    67.89369875679701,
    68.48589118007624,
    69.59357347349811,
    70.4631047778161,
    58.08023220356167,
    59.92954385791573,
    60.62655787109149,
    60.16825492695469,
    64.707281107521,
    64.35287746010036,
    65.24984563700049,
    63.926290272580694,
    69.36874809060131,
    68.78126342261892,
    69.54133894302144,
    67.96358453693023
  ],
  "8": [
    47.39266046785264,
    47.08534908049819,
    46.424865426405184,
    48.69065154043631,
    48.2388639732357,
    47.69276294665442,
    47.5157226566111,
    45.73671477977194,
    44.86171900990026,
    47.592707389879585,
    47.02899118211966,
    47.41462283076152,
    47.231127351960396,
    47.28273881568171,
    47.423905367216406,
    39.700967479378754,
    40.83714731894533,
    40.34968685917827,
    43.80681695194832,
    42.29396563644397,
    40.64490048481263,
    42.69971118640642,
    41.63442373209712,
    41.90063912427468,
    42.27761246741945,
    42.571860436667365,
    41.46993407014524,
    42.21376181732054,
    42.30325550867519,
    41.97024195403839,
    43.726127248401895,
    42.265024030297575,
    41.607759247703655,
    42.33081632744801,
    42.841554402615195,
    42.79916629472505,
    42.73151247758041,
    43.36458866607612,
    44.558026539826606,
    41.7508215346394,
    41.77176624111512,
    44.354325449511634,
    43.36469452152805,
    42.72686857694279,
    41.713950188442304,
    37.92741842437125,
    37.698800662367404,
    37.03375113704027,
    38.18995529641429,
    36.83712287275657,
    39.03118587133317,
    38.42276817658468,
    38.54725270897896,
    38.15617813732868,
    37.545323462993345,
    40.67753823711424,
    38.77235260005066,
    38.572082139268645,
    38.05303367924712,
    37.612686696894,
    43.2400748993543,
    44.54392022055761,
    42.613730655223975,
    42.96574936657389,
    42.654966882680775,
    44.79824810500728,
    43.394601064997424,
    41.03711767054019,
    41.81071066173779,
    43.63001165716799,
    41.73114958476691,
    40.93169942512528,
    43.739111070195925,
    42.096179751234,
    43.19615986501456,
    44.24216346824882,
    44.8392773907675,
    46.25549125972549,
    45.77819017356126,
    44.4226070078162,
    45.14934947094782,
    44.376546341843245,
    44.54999164804279,
    45.58678665783375,
    44.333083062237016,
    46.144530417169285,
    45.90071848901189,
    44.45414495559022,
    44.73536335996918,
    44.42353294969596,
    76.25211641361692,
    78.81129962140321,
    77.29247954283393,
    76.80174659379384,
    78.09721443470845,
    80.27607213235581,
    77.64164848558455,
    78.25838515931417,
    76.47470305017815,
    78.41127220783926,
    78.69805317850407,
    78.69307401649837,
    78.52983608173766,
    77.01998445878063,
    78.4786616537351,
    74.6954260170807,
    76.27012727350383,
    75.88900078445943,
    74.8273239308008,
    75.48349548741707,
    77.18684749480808,
    75.11309217491488,
    75.56257293821268,
    76.97424434540689,
    75.26130864681915,
    75.85911915031024,
    76.72427029567456,
    77.44847940774746,
    75.55181155924147,
    75.26119852285555,
    75.19756429090933,
    74.58165740677815,
    73.94143567044425,
    73.25950579584317,
    71.70700399251191,
    75.95147906604012,
    74.2956144067641,
    74.55287655195576,
    75.10820418066149,
    73.1025427815435,
    73.20153700062099,
    73.46431266823195,
    73.78760353780451,
    73.43935886117673,
    72.64244497399874,
    72.49331974443069,
    73.32649137054479,
    70.85016736601526,
    73.82664937754781,
    72.15992349321264,
    73.41714489269198,
    72.78144018187649,
    74.52017924642605,
    72.69608195061201,
    72.40652340791428,
    73.53315515304968,
    73.04886462348314,
    72.51267124535173,
    72.37122988059122,
    74.22717721421442
  ],
  "9": [
    72.83765824170746,
    72.88809535023199,
    63.2557045602199,
    64.62378460570825,
    64.32254051467642,
    64.22118834087676,
    63.2444069424437,
    64.66588532934253,
    66.6092227772047,
    64.36612452124272,
    61.98982469398305,
    75.86994044898765,
    74.08696386853799,
    77.57674958338475,
    73.58486048984373,
    76.21061580625502,
    73.56273020889941,
    75.94926189349265,
    76.00131916002465,
    76.41511337468557,
    75.7865757867771,
    72.4665845157914,
    73.09597573294788,
    74.55480777330277,
    74.99934008018523,
    74.01144359827912,
    75.2049838551355,
    74.22767447812312,
    75.50560723086949,
    75.71214340307345,
    75.87522268188582,
    76.03170182733251,
    75.25787433784932,
    75.25479945042133,
    75.73761795155092,
    75.60719682953518,
    76.84583009904782,
    75.28067399066487,
    75.35348991836956,
    75.99785144094459,
    75.92895083843015,
    75.49890768651359,
    75.48691865436834,
    73.58275296476744,
    74.57821120941675,
    75.8454324524905,
    75.15874265252604,
    75.15045879409301,
    74.23650410332657,
    76.82672794136221,
    74.78669314597619,
    76.63527000323165,
    75.11565902690872,
    75.37898169975587,
    75.69051992748564,
    76.72759746142484,
    76.15152241315315,
    76.72252738582979,
    77.59939201455848,
    75.7821342334519,
    76.62576128695045,
    71.59931439020974,
    71.49708037409255,
    72.38051467047704,
    69.43749904459634,
    71.0682299330144,
    72.13432778949144,
    71.90674946221293,
    72.5526846568543,
    70.20010828557446,
    71.2598845805111,
    73.33093144362516,
    72.95118058013918,
    74.27742439231392,
    72.80925485129278,
    72.70310276517688,
    72.48671963854719,
    71.72206069126014,
    73.40438023398696,
    76.79986527942576,
    76.05521193475859,
    71.1890962100779,
    73.15226674533888,
    72.38268434950363,
    72.63767099802558,
    72.96011758189186,
    71.96823981647837,
    72.37412899879419,
    73.14201092631625,
    72.39549018187347,
    67.56564211134408,
    72.33377609508109,
    68.81251055618633,
    69.46224290626296,
    68.33271755156582,
    69.40427507082521,
    69.74033789119223,
    68.96451010240231,
    67.46947014475823,
    68.32822676844091,
    70.95687711720808,
    69.40219240682248,
    68.71366214158014,
    69.54544461811206,
    67.84943827908387,
    67.45297392815299,
    70.36464905501705,
    68.20133091125393,
    69.90334278969377,
    61.90796788395396,
    59.946223673735055,
    58.93554719678754,
    61.34465571467686,
    59.44861480248551,
    60.54997460672383,
    58.86681875374959,
    61.6387443788793,
    59.887753911286616,
    60.39333850636637,
    64.97343829058727,
    62.868087110444286,
    64.87404765227879,
    63.69217310294217,
    64.21052099883133,
    62.94460832078429,
    64.95497173435223,
    63.793590916102396,
    63.74665874712481,
    63.59877845395555,
    69.15597612289119,
    70.10119166103296,
    69.51185450619764,
    69.98647138262574,
    68.75814760792282,
    69.67587821547575,
    68.13029905086852,
    67.35972385888918,
    67.49756577363529,
    68.28768402451074,
    71.0112681276427,
    70.9852138942958,
    68.41620182726271,
    71.3178163669773,
    71.25475665254837,
    70.84062986626425,
    72.13817735560163,
    71.69052293577437,
    70.75639379913977,
    69.52773618407073,
    72.57051116176126,
    71.41141149010781,
    71.54349718954492,
    69.26719543668472,
    70.06434177309288,
    71.12484763419728,
    70.14134378418537,
    71.51871992286095,
    70.51590077501778,
    64.28984215445232,
    64.29787373145682,
    65.0949119202861,
    65.29069951206486,
    63.91188407441522,
    62.85605541347989,
    64.16064323774711,
    62.599428614279,
    63.61108117839231,
    71.7186818275628,
    73.68073092820767,
    71.64178519837574,
    72.69251164547093,
    71.45861230242448,
    71.38501129315783,
    73.16089720050833,
    71.22125113261554,
    73.20354792689123,
    72.84817441348218,
    72.34183690920564,
    70.75229149260448,
    71.17749756793522,
    70.40242703790888,
    73.04579993331188,
    72.23346382947824,
    70.93065823857658,
    71.61120819712156,
    71.47802862597855,
    68.6503816976522,
    68.09763812194444,
    70.8025712057636,
    69.11992885043387
  ],
  "10": [
    75.88010171576963,
    76.67809804094624,
    77.74951825608213,
    75.2419976975451,
    77.39755670808576,
    77.11964196236214,
    76.18510598779731,
    77.62015530744351,
    76.75956004536023,
    75.16479236911941,
    75.96478672986466,
    72.44433017812106,
    74.75516740960438,
    74.57040943237413,
    75.1430672602555,
    74.66399215137127,
    72.93330626324463,
    74.71822182227274,
    75.36056970109671,
    75.05455345027725,
    73.34863270207161,
    73.82753869031887,
    72.02558615621655,
    73.94489174597226,
    73.04907230096882,
    73.2986999481178,
    73.63635444671212,
    73.51653870848878,
    71.96760060724905,
    72.60861019352431,
    73.40466595028849,
    74.26227687527485,
    71.3092435025238,
    70.04355637710272,
    72.07797925103054,
    72.18115518516912,
    71.7350347113136,
    70.07692496122928,
    72.56363393998886,
    71.89992532141689,
    70.59411496717883,
    71.26630156674649,
    72.53009811967598,
    71.3276395655976,
    70.37765067039979,
    68.55065474861765,
    68.73531985108285,
    70.73583923287823,
    67.73336277563244,
    69.41125770908721,
    68.10833535097248,
    67.85015273402392,
    69.47405186417136,
    68.16069559718024,
    66.89528874667558,
    71.11604534418886,
    69.71472690148583,
    69.70420719160505,
    70.93644002800124,
    70.77260998015046,
    71.96750985889047,
    70.94472572997186,
    71.0114254031058,
    71.17339096780634,
    70.40482371393982,
    71.47333245944107,
    73.38249544556162,
    70.3984681983075,
    72.7904301330167,
    71.87548187244,
    73.79676749565337,
    72.44656141115635,
    72.90204574010075,
    73.4203356028017,
    72.59526877852932,
    72.92436688896893,
    72.76581559323945,
    74.22481695521859,
    76.73235000200977,
    73.43956569686958,
    74.76924009025082,
    74.31632435522634,
    73.41957645833881,
    74.52689334391654,
    74.38942860245952,
    72.11693369431929,
    73.00270023199309,
    75.0921134974232,
    73.19761670992561,
    72.25904947641045,
    73.55245467388691,
    73.35063120076802,
    74.46142121728647,
    71.80159140277068,
    73.84206442471861,
    70.68594397677131,
    72.8388562306022,
    72.93953216485441,
    72.8535107235397,
    72.28784878661735,
    70.42627770136485,
    69.99054157655416,
    68.74292984484975,
    70.08432960557307,
    72.46167505552263,
    71.68289909475982,
    69.58264946449152,
    69.8238081825162,
    72.40615665439422,
    71.94068245593346
  ]
}
